In [14]:
import pandas as pd
import numpy as np

# load in merged dataset from 2021-2023
df = pd.read_csv("../data/cleaned/nyts_2021_2023_clean.csv")
print("Dataset shape:", df.shape)
print("\nClass balance:")
print(df["ecig_current"].value_counts(normalize=True))


Dataset shape: (12588, 26)

Class balance:
ecig_current
0    0.547029
1    0.452971
Name: proportion, dtype: float64


In [ ]:
# define features and target variable
feature_names = df.drop(columns=["usage", "ecig_current", "year"]).columns.tolist() # in case needed later
X = df.drop(columns=["usage", "ecig_current", "year"]).values
y = df["ecig_current"].values.reshape(-1, 1)

print("\nFeature matrix shape:", X.shape)
print("Target vector shape:", y.shape)


Feature matrix shape: (12588, 23)
Target vector shape: (12588, 1)


In [ ]:
# manual train test split 80/ 20
np.random.seed(42)

n = X.shape[0]
indices = np.random.permutation(n)

test_size = int(0.2 * n)
test_idx = indices[:test_size]
train_idx = indices[test_size:]

X_train = X[train_idx]
X_test = X[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

# standardize the data
train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)

# avoid dividing by 0
train_std[train_std == 0] = 1

X_train = (X_train - train_mean) / train_std
X_test = (X_test - train_mean) / train_std


In [18]:
# create an mlp hleper functions
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-8
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

In [19]:
# build out mlp
np.random.seed(42)

input_dim = X_train.shape[1]
hidden1_dim = 32
hidden2_dim = 16
output_dim = 1

W1 = np.random.randn(input_dim, hidden1_dim) * 0.01
b1 = np.zeros((1, hidden1_dim))

W2 = np.random.randn(hidden1_dim, hidden2_dim) * 0.01
b2 = np.zeros((1, hidden2_dim))

W3 = np.random.randn(hidden2_dim, output_dim) * 0.01
b3 = np.zeros((1, output_dim))

In [20]:
learning_rate = 0.01
epochs = 500
batch_size = 64
n_train = X_train.shape[0]

for epoch in range(epochs):
    indices = np.random.permutation(n_train)
    X_train_shuffled = X_train[indices]
    y_train_shuffled = y_train[indices]

    epoch_loss = 0

    for start in range(0, n_train, batch_size):
        end = start + batch_size
        X_batch = X_train_shuffled[start:end]
        y_batch = y_train_shuffled[start:end]

        # forward pass
        Z1 = X_batch @ W1 + b1
        A1 = relu(Z1)

        Z2 = A1 @ W2 + b2
        A2 = relu(Z2)

        Z3 = A2 @ W3 + b3
        A3 = sigmoid(Z3)

        loss = binary_cross_entropy(y_batch, A3)
        epoch_loss += loss * len(X_batch)

        # backprop
        m = len(X_batch)

        dZ3 = A3 - y_batch
        dW3 = (A2.T @ dZ3) / m
        db3 = np.sum(dZ3, axis=0, keepdims=True) / m

        dA2 = dZ3 @ W3.T
        dZ2 = dA2 * relu_derivative(Z2)
        dW2 = (A1.T @ dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m

        dA1 = dZ2 @ W2.T
        dZ1 = dA1 * relu_derivative(Z1)
        dW1 = (X_batch.T @ dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        # gradient descent update
        W3 -= learning_rate * dW3
        b3 -= learning_rate * db3

        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

    epoch_loss /= n_train

    if epoch % 50 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch}: loss = {epoch_loss:.4f}")

Epoch 0: loss = 0.6918
Epoch 50: loss = 0.6886
Epoch 100: loss = 0.6884
Epoch 150: loss = 0.6514
Epoch 200: loss = 0.6354
Epoch 250: loss = 0.6255
Epoch 300: loss = 0.6149
Epoch 350: loss = 0.6087
Epoch 400: loss = 0.6041
Epoch 450: loss = 0.5992
Epoch 499: loss = 0.5951


In [ ]:
# make some predictions with the different activation functions
Z1_test = X_test @ W1 + b1
A1_test = relu(Z1_test)

Z2_test = A1_test @ W2 + b2
A2_test = relu(Z2_test)

Z3_test = A2_test @ W3 + b3
y_prob = sigmoid(Z3_test)
y_pred = (y_prob >= 0.5).astype(int)

Accuracy: 0.6368692888359158
Confusion Matrix:
[[994 377]
 [537 609]]


In [ ]:
# look at various metrics including accuracy, confusion matrix, precision, recall, F1, etc. yk the vibes

# accuracy
accuracy = np.mean(y_pred == y_test)
print("Accuracy:", accuracy)

# confusion matrix
tp = np.sum((y_test == 1) & (y_pred == 1))
tn = np.sum((y_test == 0) & (y_pred == 0))
fp = np.sum((y_test == 0) & (y_pred == 1))
fn = np.sum((y_test == 1) & (y_pred == 0))
print("Confusion Matrix:")
print(np.array([[tn, fp],
                [fn, tp]]))

# precision, recall, f1
precision = tp / (tp + fp + 1e-8)
recall = tp / (tp + fn + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.6368692888359158
Confusion Matrix:
[[994 377]
 [537 609]]
Precision: 0.6176470588172652
Recall: 0.531413612560808
F1 Score: 0.5712945541222383
